# 🚀 Google Colab GPU & Machine Learning 实验工作台 (AI Ultra 5x ~1,000 CU/月)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davidshizw/stock-colab/blob/main/notebooks/02_Colab_GPU_ML_Factor_Ranking_and_Regimes.ipynb)

本 Notebook 对应平台 **第 5 节（Machine Learning and GPU Research）**，严格遵循：
1. **从简单统计基线（Ridge L2）开始**，再引入非线性梯度提升树（GBDT / LightGBM）多因子交互建模。
2. **Purged & Embargoed 时序切分**：严格隔离训练集（In-Sample）、验证集（Validation）与样本外测试集（Out-of-Sample），防止重叠收益窗口泄露。
3. **算力预算监控与 Google Drive Checkpoint 持久化**：自动检测 Colab GPU（T4 / L4 / A100）并追踪每月 1,000 Compute Units 消耗。

In [ ]:
# 1. 检测 Google Colab GPU 硬件与算力消耗率 (支持私有仓库拉取与 Google Drive 缓存)
import os, subprocess, sys
from pathlib import Path
from datetime import date, timedelta

REPO_SLUG = 'davidshizw/stock-colab'
if 'google.colab' in sys.modules:
    from google.colab import drive, userdata
    drive.mount('/content/drive', force_remount=False)
    drive_repo = Path('/content/drive/MyDrive/stock-colab')
    local_repo = Path('/content/stock-colab')

    if not Path('pyproject.toml').exists():
        gh_token = None
        try:
            gh_token = userdata.get('GITHUB_TOKEN')
        except Exception:
            pass
        if not gh_token and not drive_repo.exists() and not local_repo.exists():
            from getpass import getpass
            gh_token = getpass('私有仓库首次拉取请输入 GitHub Personal Access Token (之后自动缓存在 Google Drive): ').strip()
        clone_url = f'https://{gh_token}@github.com/{REPO_SLUG}.git' if gh_token else f'https://github.com/{REPO_SLUG}.git'
        target_dir = drive_repo if Path('/content/drive/MyDrive').exists() else local_repo
        if not (target_dir / 'pyproject.toml').exists():
            subprocess.run(['git', 'clone', clone_url, str(target_dir)], check=False)
        if (target_dir / 'pyproject.toml').exists():
            os.chdir(target_dir)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[ml]'], check=False)

if Path('src').exists() and str(Path('src').resolve()) not in sys.path:
    sys.path.insert(0, str(Path('src').resolve()))

from quant_platform.colab.runtime import ColabSessionManager, detect_accelerator
from quant_platform.data.store import MarketDataStore
from quant_platform.data.ingestion import build_default_pipeline
from quant_platform.colab.ml_experiments import ColabMLExperimentRunner

colab_mgr = ColabSessionManager(drive_folder_name='stock_colab_store', monthly_cu_budget=1000.0)
print('Hardware Profile:', detect_accelerator())
print('Compute Unit Budget:', colab_mgr.get_compute_unit_status())

In [ ]:
# 2. 加载或初始化 DuckDB + Parquet 跨市场面板数据
store = MarketDataStore(db_path=colab_mgr.duckdb_path, parquet_dir=colab_mgr.parquet_dir)
end_date = date(2026, 11, 6)
start_date = end_date - timedelta(days=500)

if store.check_data_quality().total_price_rows == 0:
    build_default_pipeline(store, mode='fixture').run_ingestion(start_date=start_date, end_date=end_date, incremental=False)

runner = ColabMLExperimentRunner(store=store, colab_session=colab_mgr, seed=42)

In [ ]:
# 3. 实验一：截面股票收益排序 (Ridge 统计基线 vs 非线性多因子交互 GBDT)
import json
ranking_exp = runner.run_stock_ranking_experiments(start_date=start_date, end_date=end_date, market='ALL')
print(json.dumps(ranking_exp, indent=2, ensure_ascii=False))

In [ ]:
# 4. 实验二：市场状态分类 (Market Regime Classification) 与 20 日前瞻波动率预测 (Volatility Forecasting)
vol_regime_exp = runner.run_volatility_and_regime_experiments(start_date=start_date, end_date=end_date, market='ALL')
print(json.dumps(vol_regime_exp, indent=2, ensure_ascii=False))